# 3dvid on Kaggle or Colab

Runs `stylize.py` (voxel / contour intro) and iw3 (half-SBS 3D) on a free GPU. The same notebook works on both; it detects which one it is on.

**Kaggle** (better for full renders: 4 CPUs, background runs)
1. Once per account: verify your phone number in Kaggle *Settings* (GPU and Internet stay greyed out until you do).
2. Right panel, *Session options*: **Accelerator = GPU T4 x2** (not P100: recent torch builds dropped its architecture) and **Internet = On**.
3. *+ Add Input*, *Upload*, *New Dataset*: drop your clip in. It appears under `/kaggle/input/`.
4. *Run All*. For a full render use *Save Version*, *Save & Run All (Commit)*: it runs in the background (up to 12 h), you can close the tab, and the files land in the version's *Output* tab.

**Colab** (fine for `--frame` / `--preview`; only 2 CPUs and no background runs, so a full clip means keeping the tab open for hours)
1. *Runtime*, *Change runtime type*: **T4 GPU**.
2. *File*, *Save a copy in Drive*, or your edits are lost.
3. Upload the clip with the folder icon on the left (it lands in `/content`), or put it in `MyDrive/3dvid/` and mount Drive.
4. *Run All*.

The GPU only speeds up the depth + mask models; the effects run on the CPUs. Iterate with `--frame` / `--preview`. The working folder is wiped when the session ends, so download what you want to keep.

In [ ]:
# --- settings -----------------------------------------------------------------
REPO_REF = "main"   # branch of github.com/mwihoti/3dvid to run
CLIP = None          # None = first video found under /kaggle/input, or a full path

# stylize.py flags. Fastest first:  --frame 4  (one PNG at 4 s)  ->  --preview 3  ->  no limit (full clip)
ARGS = "--preview 3 --effect voxel --bg plasma --layout wide"

MAKE_3D = False      # also render a half-SBS 3D video with iw3 (same flags as convert3d.sh)

In [ ]:
# --- setup: idempotent, re-running only pulls the latest code -----------------
import os
ON_KAGGLE = os.path.isdir("/kaggle/input")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
GPU_HINT = "Session options -> Accelerator -> GPU T4 x2" if ON_KAGGLE else "Runtime -> Change runtime type -> T4 GPU"
print("platform:", "Kaggle" if ON_KAGGLE else "Colab")
%cd {WORK}
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU: {GPU_HINT}, then rerun"
!which ffmpeg >/dev/null || (apt-get -qq update && apt-get -qq install -y ffmpeg >/dev/null)
!test -d 3dvid || git clone -q https://github.com/mwihoti/3dvid.git
!git -C 3dvid fetch -q origin && git -C 3dvid checkout -q -B {REPO_REF} origin/{REPO_REF}
# nunif (iw3) at the commit pinned in .gitmodules
!git -C 3dvid submodule update --init -q
# torch, numpy, opencv and scikit-learn come preinstalled with CUDA; don't reinstall them.
# rembg[gpu] brings onnxruntime-gpu, which must not sit next to the CPU onnxruntime.
!pip uninstall -y -q onnxruntime 2>/dev/null; pip install -q -r 3dvid/nunif/requirements.txt "rembg[gpu]>=2.0.60"

In [ ]:
# --- sanity check -------------------------------------------------------------
import torch, onnxruntime as ort
if torch.cuda.is_available():
    print("torch", torch.__version__, "on", torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
    if torch.cuda.get_device_capability(0) < (7, 0):
        print("WARNING: this GPU is too old for current torch; switch to a T4: " + GPU_HINT)
else:
    print("WARNING: torch sees no GPU, depth will run on CPU (slow). " + GPU_HINT)
# CUDAExecutionProvider here means the person mask runs on the GPU too; if it is missing
# rembg silently falls back to CPU, which still works but makes the mask pass ~10x slower.
print("onnxruntime", ort.__version__, ort.get_available_providers())

In [ ]:
# --- pick the clip --------------------------------------------------------------
import glob, re
VIDEO_EXT = (".mp4", ".mov", ".m4v", ".mkv", ".webm", ".avi")
OUT_DIR = os.path.join(WORK, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)
if CLIP is None:
    if ON_KAGGLE:
        candidates = glob.glob("/kaggle/input/**/*", recursive=True)
    else:
        # top level of /content (the Files panel upload) + MyDrive/3dvid if Drive is mounted;
        # never the outputs folder, or a rerun would pick up its own render
        candidates = glob.glob("/content/*") + glob.glob("/content/drive/MyDrive/3dvid/**/*", recursive=True)
    found = sorted(p for p in candidates if p.lower().endswith(VIDEO_EXT))
    where = "+ Add Input -> Upload -> New Dataset" if ON_KAGGLE else "the folder icon on the left -> upload (or MyDrive/3dvid/)"
    assert found, f"No video found: upload your clip via {where}, then rerun this cell"
    CLIP = found[0]
    if len(found) > 1:
        print("other clips (set CLIP to use one):", *found[1:], sep="\n  ")
print("clip:", CLIP)

# depth + mask cache: re-renders of the same clip in this session skip both models
os.environ["STYLIZE_CACHE_DIR"] = os.path.join(WORK, "cache")
name = os.path.splitext(os.path.basename(CLIP))[0]
# name outputs after the flags so different settings don't overwrite each other
OUT_BASE = os.path.join(OUT_DIR, name + "_" + re.sub(r"[^a-z0-9]+", "_", ARGS.lower()).strip("_")[:60])

In [ ]:
# --- render -----------------------------------------------------------------------
!cd {WORK}/3dvid && python stylize.py "{CLIP}" {ARGS} -o "{OUT_BASE}"

In [ ]:
# --- optional: half-SBS 3D for a VR headset (mirrors convert3d.sh, but on the GPU) ---
if MAKE_3D:
    out3d = os.path.join(OUT_DIR, name + "_3d.mp4")
    get_ipython().system(f'cd {WORK}/3dvid/nunif && python -m iw3.cli --gpu 0 --depth-model Any_V2_S '
                         f'--divergence 2.0 --convergence 0.5 --half-sbs --max-output-height 720 '
                         f'-i "{CLIP}" -o "{out3d}" --yes')

In [ ]:
# --- show the results -----------------------------------------------------------------
from IPython.display import Video, Image, display
files = sorted(glob.glob(OUT_DIR + "/*"), key=os.path.getmtime)
for f in files:
    print(f"{os.path.getsize(f) / 1e6:7.1f} MB  {f}")
newest = [f for f in files if f.startswith(OUT_BASE)]
for f in newest:
    if f.endswith(".png"):
        display(Image(f, width=480))
    elif f.endswith(".mp4"):
        display(Video(f, embed=True, width=480))
print("Download: Output panel on the right (or the version's Output tab after a commit run)" if ON_KAGGLE
      else "Download: folder icon on the left -> outputs -> right-click -> Download")